In [1]:
import torch

import torch.nn as nn
%run model.ipynb

from timm.layers import SelectAdaptivePool2d , GELUTanh


cuda


In [2]:
#
# def efficientvit_backbone_l1(**kwargs) -> EfficientViTBackboneLarge:
#     backbone = EfficientViTBackboneLarge(
#         width=[32, 64, 128, 256, 512],
#         depth=[1, 1, 1, 6, 6],
#     )
#     return backbone
# def make_block(self, arg_dict):
#     if arg_dict['block'] == "conv":
#         blc = ConvLayer(
#             in_channels=arg_dict['in_channels'],
#             out_channels=arg_dict['out_channels'],
#             stride=arg_dict['stride'],
#             norm=None,
#             use_bias=False,
#             activation=None,
#         )
#     elif arg_dict['block'] == "res":
#         blc = ResBlock(
#             in_channels=arg_dict['in_channels'],
#             out_channels=arg_dict['out_channels'],
#             stride=arg_dict['stride'],
#             use_bias=(True, False) if arg_dict.get('fewer_norm', False) else False,
#             norm=(None, arg_dict['norm']) if arg_dict.get('fewer_norm', False) else arg_dict['norm'],
#             activation=(arg_dict['activation'], None),
#         )
#     elif arg_dict['block'] == "fmb":
#         blc = FusedMobileBottleneckConvLayer(
#             in_channels=arg_dict['in_channels'],
#             out_channels=arg_dict['out_channels'],
#             stride=arg_dict['stride'],
#             expand_ratio=arg_dict['expand_ratio'],
#             use_bias=(True, False) if arg_dict.get('fewer_norm', False) else False,
#             norm=(None, arg_dict['norm']) if arg_dict.get('fewer_norm', False) else arg_dict['norm'],
#             activation=(arg_dict['activation'], None),
#         )
#     elif arg_dict['block'] == "mb":
#         blc = MobileBottleneckConvLayer(
#             in_channels=arg_dict['in_channels'],
#             out_channels=arg_dict['out_channels'],
#             stride=arg_dict['stride'],
#             expand_ratio=arg_dict['expand_ratio'],
#             use_bias=(True, True, False) if arg_dict.get('fewer_norm', False) else False,
#             norm=(None, None, arg_dict['norm']) if arg_dict.get('fewer_norm', False) else arg_dict['norm'],
#             activation=(arg_dict['activation'], arg_dict['activation'], None),
#         )
#     elif arg_dict['block'] == 'eff':
#         blc = EfficientViTModule(
#             in_channels=arg_dict['in_channels'],
#             dimention=arg_dict['dimention'],
#             expand_ratio=arg_dict['expand_ratio'],
#             scales=arg_dict['scales'],
#             norm=arg_dict['norm'],
#             activation=arg_dict['activation'],
#         )
#     return blc

        

# def make_modules(block , residual = True):
#     all = []
#     for i in range(len(block)):
#         all.append(ResidualBlock(self.make_block(block[i]), IdentityLayer()))
#     return all

%run model.ipynb
class Stem(nn.Sequential):
    def __init__(self,
                stem_args):
        super().__init__()

        
        
        self.add_module('cov0' , EfficientViTBackboneLarge.make_block(self = None , arg_dict = stem_args['conv0']))
        self.add_module('res' , ResidualBlock(EfficientViTBackboneLarge.make_block(None , stem_args['res0']), IdentityLayer()))


class Classifier(nn.Module):
    def __init__(self,
                classifier_args):
        super(Classifier, self).__init__()

        self.conv = EfficientViTBackboneLarge.make_block(None , classifier_args['conv'])
        self.global_pool = SelectAdaptivePool2d(pool_type=classifier_args['pool']['type'], flatten=True)
        # self.classify = nn.Sequential( EfficientViTBackboneLarge.make_block( None ,classifier_args['Linear0']), EfficientViTBackboneLarge.make_block(None ,classifier_args['Linear1']))
        self.classifier = nn.Sequential(
            nn.Linear(classifier_args['Linear0']['input_shape'], classifier_args['Linear0']['output_shape'], bias=False),
            nn.LayerNorm(classifier_args['Linear0']['output_shape'], eps=1e-5),
            classifier_args['Linear0']['activation'](inplace=True) if classifier_args['Linear0'] is not None else nn.Identity(),
            nn.Dropout(classifier_args['Linear0']['dropout'], inplace=False),
            nn.Linear(classifier_args['Linear1']['input_shape'], classifier_args['Linear1']['output_shape'], bias=True),
            nn.Softmax()
        )


    def forward(self, x):
        x = self.conv(x)
        x = self.global_pool(x)

       
        return self.classifier(x)
        
class EfficientViTBackboneLarge(nn.Module):
    def __init__(self,
                width=[32, 64, 128, 256, 512],
                depth=[1, 1, 1, 6, 6] ,
                
                expand_list  = [1, 4, 4, 4, 6],
                in_channels=3,
                fewer_norm_list  = [False, False, False, True, True],
                qkv_dim=32,
                norm= nn.BatchNorm2d,
                activation=GELUTanh,
            
                ):
        super(EfficientViTBackboneLarge, self).__init__()
        

        

        stem_args = { 'conv0' : {
            'block': 'conv',
            'in_channels': 3,
            'out_channels': width[0],
            'stride': 2,
            'norm': nn.BatchNorm2d,
            'activation': GELUTanh
        },

                      'res0' : {
            'block': 'res',
            'in_channels': width[0],
            'out_channels': width[0],
            'stride': 1,
            'expand_ratio': expand_list[0],
            'norm': norm,
            'activation': activation,
            'fewer_norm': fewer_norm_list[0]}}

        self.stem = Stem(stem_args)

        
        # For res0
        # res0 = self.stage.append(ResidualBlock(
        #     self.make_block({
        #         'block': 'res',
        #         'in_channels': width[0],
        #         'out_channels': width[0],
        #         'stride': 1,
        #         'expand_ratio': expand_list[0],
        #         'norm': norm,
        #         'activation': activation,
        #         'fewer_norm': fewer_norm_list[0]
        #     }),
        #     IdentityLayer()
        # ))
        # self.stem.append(conv0)
        # self.stem.append(res0)
        
        #STEM DONE

        #woking on VITblocks
        
        

        

        # Define arguments for EfficintViTModule0
        # Define arguments for EfficientViTModule0
        
# Define arguments for EfficientViTModule0
        # args_list_module_00 = [{
        #     'block': 'my',
        #     'in_channels': width[0],
        #     'activation' : activation
        # },{
        #     'block': 'my',
        #     'in_channels': width[0],
        #     'activation' : activation
        # },{
        #     'block': 'my',
        #     'in_channels': width[0],
        #     'activation' : activation
        # },{
        #     'block': 'my',
        #     'in_channels': width[0],
        #     'activation' : activation
        # }]
        args_list_module0 = [{
            'block': 'fmbconv',
            'in_channels': width[0],
            'out_channels': width[1],
            'stride': 2,
            'expand_ratio': expand_list[1] * 4,
            'scales': 3,
            'norm': norm,
            'activation': activation
        },{
            'block': 'fmbconv',
            'in_channels': width[1],
            'out_channels': width[1],
            'stride': 1,
            'expand_ratio': expand_list[1],
            'scales': 3,
            'norm': norm,
            'activation': activation
        }]
        
        # Define arguments for EfficientViTModule1
        args_list_module1 = [{
            'block': 'fmbconv',
            'in_channels': width[1],
            'out_channels': width[2],
            'stride': 2,
            'expand_ratio': expand_list[1] * 4,
            'scales': 3,
            'norm': norm,
            'activation': activation
        }, {
            'block': 'fmbconv',
            'in_channels': width[2],
            'out_channels': width[2],
            'stride': 1,
            'expand_ratio': expand_list[1],
            'scales': 3,
            'norm': norm,
            'activation': activation
        }]
        
        # Define arguments for EfficientViTModule2
        args_list_module2 = [{
            'block': 'mbconv',
            'in_channels': width[2],
            'out_channels': width[3],
            'stride': 2,
            'expand_ratio': expand_list[2],
            'scales': 3,
            'norm': norm,
            'activation': activation
        },{
            'block': 'my',
            'in_channels': width[3],
            'activation' : activation
        },{
            'block': 'mbconv',
            'in_channels': width[3],
            'out_channels': width[3],
            'stride': 1,
            'expand_ratio': expand_list[2],
            'scales': 3,
            'norm': norm,
            'activation': activation
        },{
            'block': 'my',
            'in_channels': width[3],
            'activation' : activation
        }, {
            'block': 'mbconv',
            'in_channels': width[3],
            'out_channels': width[3],
            'stride': 1,
            'expand_ratio': expand_list[2],
            'scales': 3,
            'norm': norm,
            'activation': activation
        },{
            'block': 'my',
            'in_channels': width[3],
            'activation' : activation
        },{
            'block': 'mbconv',
            'in_channels': width[3],
            'out_channels': width[3],
            'stride': 1,
            'expand_ratio': expand_list[2],
            'scales': 3,
            'norm': norm,
            'activation': activation
        },{
            'block': 'my',
            'in_channels': width[3],
            'activation' : activation
        }, {
            'block': 'mbconv',
            'in_channels': width[3],
            'out_channels': width[3],
            'stride': 1,
            'expand_ratio': expand_list[2],
            'scales': 3,
            'norm':norm,
            'activation': activation
        },{
            'block': 'my',
            'in_channels': width[3],
            'activation' : activation
        },{
            'block': 'mbconv',
            'in_channels': width[3],
            'out_channels': width[3],
            'stride': 1,
            'expand_ratio': expand_list[2],
            'scales': 3,
            'norm': norm,
            'activation': activation
         }] #,{
        #     'block': 'my',
        #     'in_channels': width[3],
        #     'activation' : activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[3],
        #     'out_channels': width[3],
        #     'stride': 1,
        #     'expand_ratio': expand_list[2],
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # }]
        
        # Define arguments for EfficientViTModule3
        args_list_module3 = [{
            'block': 'mbconv',
            'in_channels': width[3],
            'out_channels': width[4],
            'stride': 2,
            'expand_ratio': expand_list[4] * 4,
            'scales': 3,
            'norm': norm,
            'activation': activation
        }]

        #diff from effViT

        args_list_module4 = [{
            'block': 'mbconv',
            'in_channels': width[4],
            'out_channels': width[4],
            'stride': 1,
            'expand_ratio': expand_list[4] * 4,
            'scales': 1,
            'norm': None,
            'activation': nn.ReLU
        },{
            'block': 'mbconv',
            'in_channels': width[4],
            'out_channels': width[4],
            'stride': 1,
            'expand_ratio': expand_list[4] * 4,
            'scales': 1,
            'norm': None,
            'activation': nn.ReLU
        },{
            'block': 'mbconv',
            'in_channels': width[4],
            'out_channels': width[4],
            'stride': 1,
            'expand_ratio': expand_list[4] * 4,
            'scales': 1,
            'norm': None,
            'activation': nn.ReLU
        }]
        #{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # }]
        
                
        # args_list_mymod = [{
        #     'block': 'my',
        #     'in_channels': width[1],
        #     'out_channels': width[1],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[1],
        #     'out_channels': width[1],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[1],
        #     'out_channels': width[1],
           
        # }]

        # args_list_mymod2 = [{
        #     'block': 'my',
        #     'in_channels': width[2],
        #     'out_channels': width[2],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[2],
        #     'out_channels': width[2],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[2],
        #     'out_channels': width[2],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[2],
        #     'out_channels': width[2],
           
        # }]


        # args_list_mymod3 = [{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # },{
        #     'block': 'my',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
           
        # }]

        # args_test_mb_trans = [{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # },{
        #     'block': 'mbconv',
        #     'in_channels': width[4],
        #     'out_channels': width[4],
        #     'stride': 1,
        #     'expand_ratio': expand_list[4] * 4,
        #     'scales': 3,
        #     'norm': norm,
        #     'activation': activation
        # }]
        
                
        self.stage = nn.Sequential()

        # spp = SPPTransformer(64,64)
        # Create the modules using the unpacked arguments
        self.EfficientViTModule3 = nn.Sequential()

    
        stage_list = []
        # print(type(self.make_res_blocks(args_list_module0)))


        # stage_list.append(self.make_res_blocks(args_list_module_00 , [False , False,False,False]))
        stage_list.append(self.make_res_blocks(args_list_module0 , [False , True ]))

        
        stage_list.append(self.make_res_blocks(args_list_module1 ,[False  ,True ])  )

        
        
        stage_list.append(self.make_res_blocks(args_list_module2 ,  [False  , False  , True ,False  , True  ,False  ,  True ,False  , True  ,False, True ]) )

        

        
        
        

        

        # print(stage_list)

        # print(*stage_list)

        self.stage.extend(stage_list)
        
   
        self.EfficientViTModule3.append( self.make_res_blocks(args_list_module3 , [False]  )) #reuquired 
        self.EfficientViTModule3.append(self.make_res_blocks(args_list_module4 , [True ,True,True]  ))
        

        # self.EfficientViTModule3.append(self.make_res_blocks(args_test_mb_trans , [False , True , True , True , True , True ] , trynew = 'yes'))
        # self.EfficientViTModule3.append(self.make_res_blocks(args_list_mymod3 ,[False, False ,False,False , False , False , False])) 

        #efficentvitmodule1

        # EfficintViTModule0 = self.make_res_blocks([[block = 'fmbconv' , in_channels=width[0] ,out_channels= width[1], stride = 2 ,expand_ratio=expand_list[1]*4,scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'fmbconv' ,in_channels=width[1] ,out_channels= width[1],stride = 1,expand_ratio=expand_list[1],scales=(3),norm=(norm,norm),
        #                     activation=activation ]])

        # EfficintViTModule1 = self.make_res_blocks([[block = 'fmbconv' , in_channels=width[1] ,out_channels= width[2], stride = 2 ,expand_ratio=expand_list[1]*4,scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'fmbconv' ,in_channels=width[2] ,out_channels= width[2],stride = 1,expand_ratio=expand_list[1],scales=(3),norm=(norm,norm),
        #                     activation=activation ]])

        # EfficintViTModule2 = self.make_res_blocks([[block = 'mbconv' , in_channels=width[2] ,out_channels= width[3], stride = 2 ,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'mbconv' ,in_channels=width[3] ,out_channels= width[3],stride = 1,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'mbconv' ,in_channels=width[3] ,out_channels= width[3],stride = 1,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'mbconv' ,in_channels=width[3] ,out_channels= width[3],stride = 1,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'mbconv' , in_channels=width[2] ,out_channels= width[3], stride = 1 ,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'mbconv' ,in_channels=width[3] ,out_channels= width[3],stride = 1,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],
        #                                             [block = 'mbconv' ,in_channels=width[3] ,out_channels= width[3],stride = 1,expand_ratio=expand_list[2],scales=(3),norm=(norm,norm),
        #                     activation=activation ],])

        # EfficintViTModule3 = self.make_res_blocks([block = 'mbconv' , in_channels=width[3] ,out_channels= width[4], stride = 2 ,expand_ratio=expand_list[4]*4,scales=(3),norm=(norm,norm),
        #                         activation=activation ])

        
        
        # att1 = self.make_block('eff' ,in_channels =width[4], dimention=qkv_dim, expand_ratio = expand_list[4] , scales=(3),
        #                         norm=(norm,norm),lllA
        #                         activation=activation, )
        # att2 = self.make_block('eff' ,in_channels =width[4], dimention=qkv_dim, expand_ratio = expand_list[4] , scales=(3),
        #                         norm=(norm,norm),
        #                         activation=activation, )


        args_list_att = [{
                'block': 'eff',
                'in_channels': width[4],
                'dimention': qkv_dim,
                'expand_ratio': expand_list[4],
                'scales': (5,),
                'norm': norm,
                'activation': activation
            },{
                'block': 'eff',
                'in_channels': width[4],
                'dimention': qkv_dim,
                'expand_ratio': expand_list[4],
                'scales': (3,),
                'norm': (norm, norm),
                'activation': activation}]
        
        # Create the attention blocks using the unpacked arguments
        # att1 = self.make_block(args_list_att[0])
        # cas = CascadedGroupAttention(512,512)

        # # print(att1)

        # for _ in range(0):
            
        #     self.EfficientViTModule3.append(ResidualBlock(cas, IdentityLayer()))
            
        # self.stage.append(self.EfficientViTModule3)

        #classifier head 

        classifier_args = {'conv' : {
                'block': 'conv',
                'in_channels': width[4],
                'out_channels': width[4] * 6,
                'stride': 1,
                'norm': norm,
                'activation': nn.ReLU},
                        'pool' : {'type' : 'avg'},
                        'Linear0' : {'block' : 'lin',
                                    'input_shape' : width[4]*6,
                                    'output_shape' : 3200,
                                    'use_bias' : False,
                                    'dropout' : 0,
                                    'norm' : nn.LayerNorm,
                                    'activation' : nn.ReLU},
                        'Linear1' : {'block' : 'lin',
                                    'input_shape' :3200,
                                    'output_shape' : 4,
                                    'use_bias' : False,
                                    'dropout' : 0,
                                    'norm' : None,
                                    'activation' : None}}


        self.head = Classifier(classifier_args)
        
        # self.classifier = [self.make_block({
        # 'block': 'conv',
        # 'in_channels': width[4],
        # 'out_channels': width[4] * 6,
        # 'stride': 1,
        # 'norm': norm,
        # 'activation': activation
        # })]

        
        # pool = SelectAdaptivePool2d(pool_type='avg', flatten=True)
        # self.classifier.append(pool)

        # seq = torch.nn.Sequential(
        #     LinearLayer(width[4]*6 , 3200 , 0 , use_bias = False),
        #     LinearLayer(3200 , 1000 , 0 ,  False , None , None)
        # )
        # self.classifier.append(seq)

        #make model
        # print(self.stem)


    def forward(self , x):
        x = self.stem(x)
        
        x = self.stage(x)

        # print("yo")
        x = self.EfficientViTModule3(x)
        return self.head(x)

    def make_block(self, arg_dict):

        # print(arg_dict)
        # print(len(arg_dict))

        
        
        if arg_dict['block'] == "conv":
            blc = ConvLayer(
                in_channels=arg_dict['in_channels'],
                out_channels=arg_dict['out_channels'],
                stride=arg_dict['stride'],
                norm=arg_dict['norm'] if arg_dict.get('norm') else None,
                use_bias=arg_dict['use_bias'] if arg_dict.get('use_bias') else False,
                activation=arg_dict['activation'] if arg_dict.get('activation') else None,)
        elif arg_dict['block'] == "res":
            # print("hello" , arg_dict['activation'])
            blc = ResBlock(
                in_channels=arg_dict['in_channels'],
                out_channels=arg_dict['out_channels'],
                stride=arg_dict['stride'],
                use_bias=(True, False) if arg_dict.get('fewer_norm', False) else False,
                norm=(None, arg_dict['norm']) if arg_dict.get('fewer_norm', False) else arg_dict['norm'],
                activation=(arg_dict['activation'], None),)
        elif arg_dict['block'] == "fmbconv":
            blc = FusedMobileBottleneckConvLayer(
                in_channels=arg_dict['in_channels'],
                out_channels=arg_dict['out_channels'],
                stride=arg_dict['stride'],
                expand_ratio=arg_dict['expand_ratio'],
                use_bias=(True, False) if arg_dict.get('fewer_norm', False) else False,
                norm=(arg_dict['norm'], arg_dict['norm']) if arg_dict.get('fewer_norm', False) else arg_dict['norm'],
                activation=(arg_dict['activation'], None),)
        elif arg_dict['block'] == "mbconv":
            
            blc = MobileBottleneckConvLayer(
                in_channels=arg_dict['in_channels'],
                out_channels=arg_dict['out_channels'],
                stride=arg_dict['stride'],
                expand_ratio=arg_dict['expand_ratio'],
                use_bias=(True, True, False) if arg_dict.get('fewer_norm', False) else False,
                norm=(None, None, arg_dict['norm']) ,
                activation=(arg_dict['activation'], arg_dict['activation'], None),)

                #if arg_dict.get('fewer_norm', False) else arg_dict['norm']
        elif arg_dict['block'] == 'eff':
            blc = EfficientViTModule(
                in_channels=arg_dict['in_channels'],
                dimention=arg_dict['dimention'],
                expand_ratio=arg_dict['expand_ratio'],
                scales=arg_dict['scales'],
                norm=arg_dict['norm'],
                activation=arg_dict['activation'],)

        elif arg_dict['block'] == 'lin':

            
            blc = LinearLayer(
                input_shape =arg_dict['input_shape'] ,
                output_shape = arg_dict['output_shape'],
               
                dropout = arg_dict['dropout'],
                use_bias = arg_dict['use_bias'],
                norm = arg_dict['norm'] or nn.LayerNorm,
                activation = arg_dict['activation']  ,
            )

        elif arg_dict['block'] == 'my':
            blc = MyModule(
                in_channels =arg_dict['in_channels'] ,
                activation = arg_dict['activation'],
               
            )
        return blc
    
            
    
    def make_res_blocks(self , blocks , residual , trynew = None ):
        all = nn.Sequential()

        if trynew == None:
            for i in range(len(blocks)):
            # print(blocks[i])
                if residual[i] == True:
                    
                    all.append(ResidualBlock(self.make_block(blocks[i]), IdentityLayer()))
                    
                else:
                    all.append(ResidualBlock(self.make_block(blocks[i]), None))
        
            return all

        elif trynew == 'yes':
        
            for i in range(len(blocks)):
    
                args_mymod = {
                    'block' : 'my',
                    'in_channels' : blocks[i]['in_channels'],
                    'out_channels' : blocks[i]['out_channels']
                }
                # print(blocks[i])
                if residual[i] == True:
                    
                    all.append(ResidualBlock(self.make_block(blocks[i]), self.make_block(args_mymod)))
                    
                else:
                    all.append(ResidualBlock(self.make_block(blocks[i]), None))
            
            return all

cuda


In [3]:
model = EfficientViTBackboneLarge()

In [4]:
model

EfficientViTBackboneLarge(
  (stem): Stem(
    (cov0): ConvLayer(
      (dropout): Dropout2d(p=0, inplace=False)
      (Convlayer): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): GELUTanh()
    )
    (res): ResidualBlock(
      (pre_norm): Identity()
      (main): ResBlock(
        (Conv1): ConvLayer(
          (dropout): Dropout2d(p=0, inplace=False)
          (Convlayer): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): GELUTanh()
        )
        (Conv2): ConvLayer(
          (dropout): Dropout2d(p=0, inplace=False)
          (Convlayer): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_